# Encoding sem Data Leakage — Fluxo Completo

Demonstra o uso de `codifica_categoricas` + `aplica_encoders` do início ao fim:

1. Criação do dataset com diferentes tipos de colunas categóricas  
2. Split treino/teste **antes** de qualquer encoding  
3. Fit dos encoders **apenas no treino**  
4. Transform do teste com os encoders já ajustados  
5. Verificação de consistência e inspeção dos encoders

| Tipo de coluna | Comportamento |
|---|---|
| Binária (2 valores) | OneHotEncoder `drop_first` → 1 coluna 0/1 |
| Nominal sem hierarquia (N valores) | OneHotEncoder → N-1 colunas |
| Ordinal com hierarquia definida | OrdinalEncoder → 0, 1, 2… |
| Alta cardinalidade (> max_cats_ohe) | Pulada com aviso |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 1. Dataset sintético

In [ ]:
from ML.preparacao_dados import split_dataset, codifica_categoricas, aplica_encoders

np.random.seed(42)
n = 200

df = pd.DataFrame({
    # Binaria (2 valores) -> OHE gera 1 coluna
    'genero':        np.random.choice(['M', 'F'], n),

    # Nominal sem hierarquia (4 valores) -> OHE gera 3 colunas
    'canal_venda':   np.random.choice(['Loja', 'Online', 'Televendas', 'Parceiro'], n),

    # Ordinal com hierarquia clara -> OrdinalEncoder com ordem definida
    'nivel_plano':   np.random.choice(['Basico', 'Intermediario', 'Premium'], n),

    # Ordinal com hierarquia -> passada como lista (ordem alfabetica)
    'risco_credito': np.random.choice(['Alto', 'Baixo', 'Medio'], n),

    # Alta cardinalidade -> sera pulada automaticamente
    'cidade':        [f'Cidade_{i}' for i in np.random.randint(1, 60, n)],

    # Numericas (passam sem alteracao)
    'idade':             np.random.randint(18, 65, n),
    'valor_mensalidade': np.round(np.random.uniform(50, 300, n), 2),

    'churn': np.random.randint(0, 2, n),
})

print(f"Shape: {df.shape}")
df.head()


## 2. Split treino / teste

> O split acontece **antes** de qualquer encoding. Fazer ao contrário causaria data leakage: o encoder aprenderia estatísticas do conjunto de teste.

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(
    data=df,
    target_column='churn',
    test_size=0.2,
    random_state=42,
)

print(f"Treino: {X_train.shape}  |  Teste: {X_test.shape}")


## 3. Encoding no treino

O fit dos encoders acontece aqui.  
- `colunas_ordinais` como **dict** → você controla a ordem exata  
- `drop_first=True` → evita dummy variable trap  
- `max_cats_ohe=20` → colunas com mais categorias são ignoradas

In [ ]:
X_train_enc, encoders = codifica_categoricas(
    df=X_train,
    colunas_ordinais={
        'nivel_plano':   ['Basico', 'Intermediario', 'Premium'],   # 0 -> 1 -> 2
        'risco_credito': ['Baixo', 'Medio', 'Alto'],               # 0 -> 1 -> 2
    },
    drop_first=True,
    max_cats_ohe=20,
)

print(f"Shape apos encoding: {X_train_enc.shape}")
X_train_enc.head()


## 4. Encoding no teste

`aplica_encoders` usa os encoders ajustados no treino — **sem refazer o fit**.  
Categorias desconhecidas no teste são tratadas com segurança:  
- OHE: preenche com 0 em todas as colunas geradas  
- OrdinalEncoder: retorna -1

In [ ]:
X_test_enc = aplica_encoders(df=X_test, encoders=encoders)

print(f"Shape apos encoding: {X_test_enc.shape}")
X_test_enc.head()


## 5. Verificação de consistência

In [ ]:
colunas_ok = X_train_enc.columns.tolist() == X_test_enc.columns.tolist()
print(f"Colunas treino == Colunas teste: {colunas_ok}")
print(f"\nColunas finais: {X_train_enc.columns.tolist()}")


## 6. Inspecionando os encoders

O dict `encoders` retornado por `codifica_categoricas` guarda tudo que `aplica_encoders` precisa para repetir a transformação em qualquer conjunto futuro.

In [ ]:
print("Encoders ajustados:")
for col, info in encoders.items():
    if info['tipo'] == 'ordinal':
        ordem = info['encoder'].categories_[0].tolist()
        print(f"  {col:<20} ORDINAL   | ordem: {ordem}")
    elif info['tipo'] == 'ohe':
        print(f"  {col:<20} OHE       | colunas: {info['colunas_geradas']}")
